# Measure parallel DICOM preparation before submission

The serial measurement was **37.04 seconds/study**, including **about 29 minutes of preparation for fifty studies**. It only projects to about 644 studies within the eight-hour internal limit. A successful serial benchmark does not establish full-test feasibility.

Attach the inference package from the successful **03b timing run** (a private dataset containing its `rsna3-inference/` folder is accepted), plus the **RSNA competition**. Select **T4 ×2**, Internet off, update the quota below, then **Save Version → Save & Run All**.

This notebook uses four independent CPU processes to call the **unchanged** renderer/cache writer. The trained weights, source hash, all windows and ensemble formula remain unchanged. Cache shards are combined without changing their pixel payloads. A separately hashed helper pins the new execution settings. The initial input search uses known Kaggle paths and does not traverse the competition DICOM tree.

It repeats the full fifty-study timing measurement with the new execution path. No speedup is assumed: check `fits_planning_scenario` and the new `admission_profile`. The 1,300-study calculation is a planning scenario; actual test IDs are always read at submission time. Keep your existing baseline submission.

After a sufficiently fast result, use **04_soumettre_parallele.ipynb** with this notebook's new output package. The original serial submission notebook cannot use the faster parallel estimate.


In [ ]:
# Update the quota if Kaggle now shows a different amount.
QUOTA_AVAILABLE = 13.0
MEASURED_QUOTA_FACTOR = 1.0
CPU_WORKERS = 4
CACHE_BATCH_SIZE = 4
MAX_BENCHMARK_HOURS = 2.0
PLANNING_STUDIES = 1300  # Planning scenario only; submission always reads actual test.csv.


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
BUNDLE_ROOT = None
if BUNDLE_ROOT is None:
    choices = set()
    # Known Kaggle input layouts; never recurse through the DICOM directory.
    for pattern in ('*/manifest.json', '*/rsna3-inference/manifest.json',
                    'datasets/*/*/manifest.json', 'datasets/*/*/rsna3-inference/manifest.json',
                    'notebooks/*/*/manifest.json', 'notebooks/*/*/rsna3-inference/manifest.json'):
        for p in Path('/kaggle/input').glob(pattern):
            try:
                if json.loads(p.read_text()).get('format') == 'rsna3-full-models-v1':
                    choices.add(p.parent)
            except (OSError, ValueError):
                pass
    choices = sorted(choices)
    if len(choices) != 1:
        raise ValueError(f'Choisir BUNDLE_ROOT parmi {choices}')
    BUNDLE_ROOT = choices[0]
BUNDLE_ROOT = Path(BUNDLE_ROOT)
# Optional Python ABI patch; only its shallow dataset metadata is searched.
WHEELS_PATCH_ROOT = None
patch_manifests = list(Path('/kaggle/input').glob('*/wheels_manifest.json'))
patch_manifests += list(Path('/kaggle/input').glob('datasets/*/*/wheels_manifest.json'))
patch_roots = [p.parent for p in patch_manifests
               if json.loads(p.read_text()).get('format') == 'rsna3-offline-wheels-patch-v1']
if len(patch_roots) > 1:
    raise ValueError(f'Attach a single wheels patch: {patch_roots}')
if patch_roots:
    WHEELS_PATCH_ROOT = patch_roots[0]
BOOTSTRAP_ROOT = WHEELS_PATCH_ROOT or BUNDLE_ROOT
module_spec = importlib.util.spec_from_file_location('kaggle_bootstrap', BOOTSTRAP_ROOT / 'kaggle_bootstrap.py')
bootstrap = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(bootstrap)
if WHEELS_PATCH_ROOT is None:
    bootstrap.install(BUNDLE_ROOT)
else:
    bootstrap.install(BUNDLE_ROOT, wheels_root=WHEELS_PATCH_ROOT)
WORK = Path('/kaggle/working')

from rsna3.common import read_json, require, atomic_json, sha256
from rsna3.inference import load_bundle
import shutil
SOURCE_BUNDLE = BUNDLE_ROOT
EXPORT = WORK / 'rsna3-inference'
require(not EXPORT.exists(), 'Use a fresh saved run: working inference bundle already exists.')
shutil.copytree(SOURCE_BUNDLE, EXPORT, ignore=shutil.ignore_patterns('__pycache__'))
manifest = load_bundle(EXPORT)
print('Fifteen exported model checkpoints and the training environment verified.')

def cli(*args):
    subprocess.run([sys.executable, '-m', 'rsna3', *map(str, args)], check=True)


In [ ]:
# This execution helper is separately hashed; rsna3 and all model weights stay unchanged.
helper_path = EXPORT / 'rsna3_parallel_inference.py'
helper_path.write_text('"""Parallel DICOM preparation around the unchanged, pinned RSNA3 renderer.\n\nThis module is outside rsna3: training code, rendering and model hashes stay\nunchanged. Its own checksum and worker count are pinned in the new benchmark.\n"""\nfrom __future__ import annotations\n\nimport multiprocessing as mp\nimport os\nimport shutil\nimport time\nfrom pathlib import Path\n\nPIPELINE = "parallel-study-cache-v1"\nSCOPE = "decode_render_parallel_cache_all_15_models_global_ranks"\n\n\ndef execution_contract(workers, batch_size):\n    from rsna3.common import require, sha256\n    require(isinstance(workers, int) and 1 <= workers <= 8, "Use one to eight CPU processes")\n    require(isinstance(batch_size, int) and 1 <= batch_size <= 8, "Use batches of one to eight studies")\n    return {"pipeline": PIPELINE, "workers": workers, "batch_size": batch_size,\n            "helper_sha256": sha256(Path(__file__))}\n\n\ndef _prepare_batch(task):\n    """Each spawned process calls the original renderer and original cache writer."""\n    from rsna3.cache import prepare\n    from rsna3.common import require\n    ids, series_csv, dicom_root, destination, centres, deadline = task\n    require(time.monotonic() < deadline, "Parallel preprocessing deadline reached")\n    manifest = prepare(ids, series_csv, dicom_root, destination, centres=centres,\n                       inference=True, max_hours=max(.001, (deadline - time.monotonic()) / 3600))\n    require(manifest["complete"], "Incomplete worker cache")\n    return str(Path(destination) / "manifest.json")\n\n\ndef _merge_worker_caches(ids, manifests, output, series_csv, centres, started,\n                         execution, limit_bytes):\n    from rsna3.cache import Cache, STORAGE\n    from rsna3.common import atomic_json, digest, read_json, require, sha256, source_hash\n    from rsna3.dicom import RENDER\n    contract = {"render": RENDER, "storage": STORAGE, "centres": centres,\n                "series_sha256": sha256(series_csv), "studies_hash": digest(sorted(ids)),\n                "source_hash": source_hash(), "fixture": False, "purpose": "inference"}\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    entries, shards, copied_bytes = {}, [], 0\n    for marker in sorted(map(Path, manifests)):\n        worker = read_json(marker)\n        worker_contract = worker["contract"]\n        require(worker.get("complete"), "Worker cache is incomplete")\n        for key, expected in contract.items():\n            if key != "studies_hash":\n                require(worker_contract.get(key) == expected, f"Worker rendering contract differs: {key}")\n        require(set(worker["entries"]).issubset(ids), "Unexpected study in a worker cache")\n        require(worker_contract["studies_hash"] == digest(sorted(worker["entries"])), "Worker study contract differs")\n        require(not set(entries).intersection(worker["entries"]), "Duplicate worker studies")\n        renamed = {}\n        for original in worker["shards"]:\n            old_name = original["file"]\n            source = marker.parent / old_name\n            require(sha256(source) == original["sha256"], "Worker shard is corrupt")\n            require(source.stat().st_size == original["bytes"], "Worker shard size differs")\n            new_name = f"shard-{len(shards):04d}.zip"\n            copied_bytes += original["bytes"]\n            require(copied_bytes + (4 << 20) < limit_bytes, "Combined cache exceeds its storage limit")\n            require(shutil.disk_usage(output).free > original["bytes"] + (256 << 20), "Insufficient merge disk space")\n            shutil.copy2(source, output / new_name)\n            renamed[old_name] = new_name\n            shards.append({**original, "file": new_name})\n        for uid, entry in worker["entries"].items():\n            entries[uid] = {**entry, "shard": renamed[entry["shard"]]}\n    require(set(entries) == set(ids), "Merged cache has incomplete study coverage")\n    for index, shard in enumerate(shards):\n        atomic_json(output / f"shard-{index:04d}.json", {\n            **shard, "contract": contract,\n            "entries": {uid: e for uid, e in entries.items() if e["shard"] == shard["file"]}})\n    manifest = {"contract": contract, "entries": entries, "shards": shards, "complete": True,\n                "bytes": copied_bytes, "session_seconds": time.monotonic() - started,\n                "execution": execution,\n                "laterality_conflict_studies": [uid for uid in ids if entries[uid].get("warnings", 0)]}\n    index_bytes = sum(p.stat().st_size for p in output.glob("shard-*.json"))\n    require(copied_bytes + index_bytes + (4 << 20) < limit_bytes, "Combined cache indexes exceed the storage limit")\n    atomic_json(output / "manifest.json", manifest)\n    reader = Cache(output, verify=True)\n    reader.close()\n    return manifest\n\n\ndef prepare_parallel(ids, series_csv, dicom_root, output, centres, workers=4,\n                     batch_size=4, max_hours=2., limit_bytes=12 << 30):\n    from rsna3.cache import Cache, STORAGE\n    from rsna3.common import digest, read_json, require, sha256, source_hash\n    from rsna3.dicom import RENDER\n    ids = list(map(str, ids))\n    require(ids and len(ids) == len(set(ids)), "Study IDs must be nonempty and unique")\n    require(centres in (2, 4) and max_hours > 0, "Invalid rendering settings")\n    execution = execution_contract(workers, batch_size)\n    output, started = Path(output), time.monotonic()\n    if (output / "manifest.json").exists():\n        manifest = read_json(output / "manifest.json")\n        expected = {"render": RENDER, "storage": STORAGE, "centres": centres,\n                    "series_sha256": sha256(series_csv), "studies_hash": digest(sorted(ids)),\n                    "source_hash": source_hash(), "fixture": False, "purpose": "inference"}\n        require(manifest["contract"] == expected and manifest.get("execution") == execution,\n                "Existing parallel cache belongs to different inputs or execution settings")\n        require(set(manifest["entries"]) == set(ids), "Existing parallel cache has incorrect coverage")\n        reader = Cache(output, verify=True)\n        reader.close()\n        return manifest\n    deadline = started + max_hours * 3600\n    parts = output.with_name(output.name + ".parts")\n    parts.mkdir(parents=True, exist_ok=True)\n    batches = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]\n    tasks = [(batch, str(series_csv), str(dicom_root), str(parts / digest(batch)[:24]), centres, deadline)\n             for batch in batches]\n    # Bound native BLAS/OpenMP thread pools in the spawned processes.\n    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):\n        os.environ[key] = "1"\n    pool = mp.get_context("spawn").Pool(processes=min(workers, len(tasks)))\n    manifests, done = [], 0\n    try:\n        iterator = pool.imap_unordered(_prepare_batch, tasks, chunksize=1)\n        for index in range(len(tasks)):\n            remaining = deadline - time.monotonic()\n            require(remaining > 0, "Parallel preprocessing deadline reached")\n            try:\n                marker = iterator.next(timeout=remaining)\n            except mp.TimeoutError as error:\n                raise ValueError("Parallel preprocessing deadline reached") from error\n            manifests.append(marker)\n            done += len(read_json(marker)["entries"])\n            print(f"Parallel cache: {done}/{len(ids)} studies ({index + 1}/{len(tasks)} batches)", flush=True)\n        pool.close()\n        pool.join()\n    except BaseException:\n        pool.terminate()\n        pool.join()\n        raise\n    require(time.monotonic() < deadline, "Parallel preprocessing deadline reached before merge")\n    manifest = _merge_worker_caches(ids, manifests, output, series_csv, centres, started, execution, limit_bytes)\n    shutil.rmtree(parts)\n    return manifest\n\n\ndef benchmark_parallel(bundle, studies_csv, series_csv, dicom_root, work,\n                       workers=4, batch_size=4, quota_factor=1., n=50, max_hours=2.):\n    import pandas as pd\n    from rsna3.common import UID, atomic_json, digest, read_json, require, sha256\n    from rsna3.inference import load_bundle, predict_bundle\n    require(50 <= n <= 100 and quota_factor > 0, "Use fifty to one hundred studies and the measured quota factor")\n    manifest = load_bundle(bundle)\n    execution = execution_contract(workers, batch_size)\n    table = pd.read_csv(studies_csv, dtype={UID: str})\n    require(table[UID].is_unique and len(table) >= n, "Insufficient unique sample studies")\n    ids = sorted(table[UID], key=digest)[:n]\n    work = Path(work)\n    work.mkdir(parents=True, exist_ok=True)\n    cache, started = work / "parallel-cache", time.monotonic()\n    require(not cache.exists() and not cache.with_name(cache.name + ".parts").exists(),\n            "A timing benchmark requires a fresh cache directory; use a new saved run")\n    deadline = started + max_hours * 3600\n    prepare_parallel(ids, series_csv, dicom_root, cache, manifest["centres"], workers, batch_size, max_hours)\n    cache_seconds = time.monotonic() - started\n    predict_bundle(bundle, pd.DataFrame({UID: ids}), cache, work, deadline)\n    elapsed = time.monotonic() - started\n    startup = sum(read_json(work / f"profile_{model}_fold{fold}.json")["load_seconds"]\n                  for model in manifest["configs"] for fold in range(5))\n    profile = {"studies": n, "uids_hash": digest(ids), "startup_seconds": startup,\n               "seconds_per_study": max(0., elapsed - startup) / n,\n               "quota_factor": quota_factor, "elapsed_seconds": elapsed,\n               "preprocessing_seconds": cache_seconds, "scope": SCOPE,\n               "execution": execution, "series_sha256": sha256(series_csv),\n               "cache_reused": False,\n               "coldness_note": "Mounted data and OS caches are not forcibly evicted; compare on the same study sample."}\n    # Retain the original serial measurement for auditing. Never admit the\n    # parallel estimate to the serial submit function.\n    if manifest.get("admission_profile", {}).get("scope") != SCOPE:\n        manifest["serial_admission_profile"] = manifest.get("admission_profile")\n    manifest["required_execution_pipeline"] = PIPELINE\n    manifest["admission_profile"] = profile\n    atomic_json(Path(bundle) / "manifest.json", manifest)\n    return profile\n\n\ndef submit_parallel(bundle, test_csv, series_csv, dicom_root, work, output_csv,\n                    max_hours=8., workers=4, batch_size=4, loader_workers=1):\n    import pandas as pd\n    from rsna3.common import LABELS, UID, atomic_csv, atomic_json, digest, require, sha256, source_hash, validate_predictions\n    from rsna3.inference import load_bundle, predict_bundle\n    output_csv, work = Path(output_csv), Path(work)\n    output_csv.unlink(missing_ok=True)\n    work.mkdir(parents=True, exist_ok=True)\n    require(0 < max_hours <= 8.25, "Keep time to write the CSV before the official runtime limit")\n    started = time.monotonic()\n    deadline = started + max_hours * 3600\n    branch_deadline = deadline - 120\n    table = pd.read_csv(test_csv, dtype={UID: str})\n    require(UID in table and len(table) and table[UID].is_unique, "Invalid test.csv")\n    manifest = load_bundle(bundle)\n    profile = manifest.get("admission_profile", {})\n    execution = execution_contract(workers, batch_size)\n    require(manifest.get("required_execution_pipeline") == PIPELINE and profile.get("scope") == SCOPE,\n            "Measure this parallel pipeline on fifty studies before submitting")\n    require(profile.get("execution") == execution, "Worker count, batch size or parallel helper differs from benchmark")\n    estimate = profile["startup_seconds"] + len(table) * profile["seconds_per_study"] * 1.2\n    require(time.monotonic() + estimate < branch_deadline,\n            f"Parallel pipeline exceeds its measured runtime budget: {estimate / 3600:.2f} hours")\n    ids = table[UID].tolist()\n    cache = work / ("parallel-test-cache-" + digest({"uids": ids, "series": sha256(series_csv),\n                    "source": source_hash(), "centres": manifest["centres"], "execution": execution})[:12])\n    prepare_parallel(ids, series_csv, dicom_root, cache, manifest["centres"], workers, batch_size,\n                     max_hours=max(.001, (branch_deadline - time.monotonic()) / 3600))\n    predicted = predict_bundle(bundle, table, cache, work, branch_deadline, loader_workers)\n    final = validate_predictions(pd.DataFrame({UID: ids, **dict(zip(LABELS, predicted.T))}), ids)[[UID] + LABELS]\n    require(time.monotonic() < deadline, "Inference deadline reached before CSV creation")\n    atomic_csv(output_csv, final)\n    receipt = {"recipe": "clean_equal_rank_three_families_five_fold_probability_means",\n               "execution": execution, "studies": len(ids), "test_csv_sha256": sha256(test_csv),\n               "submission_sha256": sha256(output_csv), "elapsed_seconds": time.monotonic() - started,\n               "projected_seconds_with_margin": estimate}\n    atomic_json(work / "submission_receipt.json", receipt)\n    return receipt\n')
sys.path.insert(0, str(EXPORT))
import os
os.environ['PYTHONPATH'] = str(EXPORT) + os.pathsep + os.environ.get('PYTHONPATH', '')
from rsna3_parallel_inference import benchmark_parallel, execution_contract, PIPELINE, SCOPE
print('Parallel preparation settings:', execution_contract(CPU_WORKERS, CACHE_BATCH_SIZE))


In [ ]:
COMPETITION_ROOT = None
if COMPETITION_ROOT is None:
    choices = set()
    for pattern in ('*/train_series.csv', 'competitions/*/train_series.csv', 'datasets/*/*/train_series.csv'):
        for p in Path('/kaggle/input').glob(pattern):
            if (p.parent / 'train.csv').is_file() and (p.parent / 'test.csv').is_file():
                choices.add(p.parent)
    require(len(choices) == 1, f'Attach one RSNA competition input; found: {sorted(choices)}')
    COMPETITION_ROOT = next(iter(choices))
COMPETITION_ROOT = Path(COMPETITION_ROOT)

def dicom_directory(split):
    choices = [COMPETITION_ROOT / name for name in (f'{split}_series', f'{split}_images', split)
               if (COMPETITION_ROOT / name).is_dir()]
    require(len(choices) == 1, f'Choose the DICOM folder for {split}: {choices}')
    return choices[0]


In [ ]:
require(QUOTA_AVAILABLE - MAX_BENCHMARK_HOURS * MEASURED_QUOTA_FACTOR - .5 >= 8,
        'Keep eight quota hours and a thirty-minute setup allowance protected.')
import torch
require(torch.cuda.is_available(), 'Select GPU T4 x2 for this measured full-pipeline run.')
previous = manifest.get('admission_profile', {})
if previous.get('scope') == SCOPE and previous.get('execution') == execution_contract(CPU_WORKERS, CACHE_BATCH_SIZE):
    profile = previous
    print('Reusing the existing measured parallel profile for the exact same execution settings.')
else:
    profile = benchmark_parallel(EXPORT, COMPETITION_ROOT / 'train.csv',
        COMPETITION_ROOT / 'train_series.csv', dicom_directory('train'),
        WORK / 'inference-benchmark-parallel', workers=CPU_WORKERS, batch_size=CACHE_BATCH_SIZE,
        quota_factor=MEASURED_QUOTA_FACTOR, n=50, max_hours=MAX_BENCHMARK_HOURS)
print(json.dumps(profile, indent=2))


In [ ]:
# Rebuild the offline manifest after recording the new helper and timing profile.
manifest = load_bundle(EXPORT)
profile = manifest['admission_profile']
require(manifest.get('required_execution_pipeline') == PIPELINE and profile.get('scope') == SCOPE,
        'Complete the parallel timing run before using its submission notebook.')
require(profile.get('execution') == execution_contract(CPU_WORKERS, CACHE_BATCH_SIZE),
        'Execution settings differ from the measured profile.')
atomic_json(EXPORT / 'offline_manifest.json', {'format': 'rsna3-inference-offline-v1',
    'files': {str(p.relative_to(EXPORT)): sha256(p) for p in EXPORT.rglob('*')
              if p.is_file() and p.name != 'offline_manifest.json' and '__pycache__' not in p.parts}})
seconds = profile['seconds_per_study']
planning_seconds = profile['startup_seconds'] + PLANNING_STUDIES * seconds * 1.2
maximum = max(0, int((8 * 3600 - 120 - profile['startup_seconds']) / (seconds * 1.2)))
summary = {'stage': 'parallel_inference_timing_complete', 'exported_models': len(manifest['files']),
           'admission_profile': profile, 'planning_studies': PLANNING_STUDIES,
           'planning_hours_with_margin': planning_seconds / 3600,
           'maximum_studies_under_eight_hours': maximum,
           'fits_planning_scenario': planning_seconds < 8 * 3600 - 120,
           'next_notebook': '04_soumettre_parallele.ipynb', 'bundle': str(EXPORT)}
atomic_json(WORK / 'parallel_inference_summary.json', summary)
print(json.dumps(summary, indent=2))
